In [8]:
# importing different packages
import pandas as pd
import pickle
import geopandas as gpd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [9]:
with open('name_to_fips.pkl', 'rb') as f:
    name_to_fips = pickle.load(f)
    print(name_to_fips)

with open('us_dict.pkl', 'rb') as f:
    us_dict = pickle.load(f)

zip_map = pd.read_csv("zip_to_county_mapping.csv", dtype={"zip_code": str, "county_fips": str})
zip_to_county_dict = dict(zip(zip_map["zip_code"], zip_map["county_fips"]))


data = {
    'FIPS': [
        '06001', '06003', '06005', '06007', '06009', '06011', '06013', '06015', '06017', '06019',
        '06021', '06023', '06025', '06027', '06029', '06031', '06033', '06035', '06037', '06039',
        '06041', '06043', '06045', '06047', '06049', '06051', '06053', '06055', '06057', '06059',
        '06061', '06063', '06065', '06067', '06069', '06071', '06073', '06075', '06077', '06079',
        '06081', '06083', '06085', '06087', '06089', '06091', '06093', '06095', '06097', '06099',
        '06101', '06103', '06105', '06107', '06109', '06111', '06113', '06115'
    ],
}

state_to_abbr = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
    "Puerto Rico": "PR",
    "Guam": "GU",
    "American Samoa": "AS",
    "U.S. Virgin Islands": "VI",
    "Northern Mariana Islands": "MP"
}

# Create the DataFrame
df_ca_counties = pd.DataFrame(data)

{'alabama': '1000', 'autauga': '1001', 'baldwin': '13009', 'barbour': '54001', 'bibb': '13021', 'blount': '47009', 'bullock': '1011', 'butler': '42019', 'calhoun': '54013', 'chambers': '48071', 'cherokee': '48073', 'chilton': '1021', 'choctaw': '40023', 'clarke': '51043', 'clay': '54015', 'cleburne': '5023', 'coffee': '47031', 'colbert': '1033', 'conecuh': '1035', 'coosa': '1037', 'covington': '28031', 'crenshaw': '1041', 'cullman': '1043', 'dale': '1045', 'dallas': '48113', 'dekalb': '47041', 'elmore': '16039', 'escambia': '12033', 'etowah': '1055', 'fayette': '54019', 'franklin': '53021', 'geneva': '1061', 'greene': '51079', 'hale': '48189', 'henry': '51089', 'houston': '48225', 'jackson': '55053', 'jefferson': '55055', 'lamar': '48277', 'lauderdale': '47097', 'lawrence': '47099', 'lee': '51105', 'limestone': '48293', 'lowndes': '28087', 'macon': '47111', 'madison': '51113', 'marengo': '1091', 'marion': '54049', 'marshall': '54051', 'mobile': '1097', 'monroe': '55081', 'montgomery': 

In [10]:
# DER Energy Storage from: https://www.energy.ca.gov/data-reports/energy-almanac/california-electricity-data/california-energy-storage-system-survey
# Has energy storage data at the commercial, residential, and utility scale, check for census tract storage data
storage = pd.read_excel("EnergyStorage_Cleaned_August2024_ada.xlsx")
storage['County'] = storage['County'].str.strip().str.lower() # Standardize county names
storage['fips'] = storage['County'].map(name_to_fips) # Calculate FIPS
storage['fips'] = storage['fips'].astype(str).str.zfill(5) # Format FIPS

# Make names more descriptive
rename_map = {
    "Utility": "utility_name",
    "Nameplate Capacity (in KW AC)": "nameplate_capacity_kw_ac",
    "Fuel Type": "fuel_type",
    "Facility City": "facility_city",
    "County": "county",
    "CAISO Flag": "caiso_flag",
    "Facility Zip": "zip_code",
    "Customer Sector": "customer_sector",
    "Approval Date": "approval_date",
}
storage = storage.rename(columns=rename_map)

# Filter only DER as being those in residential or commercial sector with less than 10 mW or 10000 kW capacity
storage_der = storage[storage["customer_sector"].isin(["Residential", "Commercial"])]
storage_der = storage_der[storage_der["nameplate_capacity_kw_ac"] <= 10000]

storage_utility = storage[storage["nameplate_capacity_kw_ac"] > 10000]

storage_der["zip_code"] = storage_der["zip_code"].astype(str).str.zfill(5).str.strip()
print(storage_der.head)

<bound method NDFrame.head of                   utility_name  nameplate_capacity_kw_ac fuel_type  \
0       Alameda Municpal Power                      1.90   Storage   
1       Alameda Municpal Power                      3.00   Storage   
2       Alameda Municpal Power                      3.30   Storage   
3       Alameda Municpal Power                      3.30   Storage   
4       Alameda Municpal Power                      3.30   Storage   
...                        ...                       ...       ...   
196545      Truckee Donner PUD                      9.30     Solar   
196546      Truckee Donner PUD                      9.60     Solar   
196547      Truckee Donner PUD                     10.08     Solar   
196548      Truckee Donner PUD                     10.16     Solar   
196549      Truckee Donner PUD                     11.86     Solar   

       facility_city   county caiso_flag zip_code customer_sector  \
0            Alameda  alameda      OTHER    94501     Reside

In [11]:
# Wind data from: https://energy.usgs.gov/uswtdb/
uswtdb_wind = pd.read_csv("USWTDB_wind_data.csv")
uswtdb_wind.rename(columns={'t_county': 'County'}, inplace=True)
uswtdb_wind['County'] = uswtdb_wind['County'].str.strip().str.lower()
uswtdb_wind["County"] = uswtdb_wind["County"].str.replace(" county", "", regex=False)
uswtdb_wind['FIPS'] = uswtdb_wind['County'].map(name_to_fips)
uswtdb_wind['FIPS'] = uswtdb_wind['FIPS'].astype(str).str.zfill(5)
uswtdb_wind = uswtdb_wind.drop(columns=["t_fips"])
rename_map = {
    # IDs & references
    "case_id":          "case_id",                       # unique row ID (keep as‑is)
    "faa_ors":          "faa_ors_id",                    # FAA Obstruction Repository System ID
    "faa_asn":          "faa_asn_id",                    # FAA Aeronautical Study Number
    "usgs_pr_id":       "usgs_project_id",
    "eia_id":           "eia_id",

    # Location
    "t_state":          "state",
    "County":           "county",
    "FIPS":             "fips",                     # duplicate field in raw file
    "xlong":            "longitude",
    "ylat":             "latitude",

    # Project‑level info
    "p_name":           "project_name",
    "p_year":           "project_year",
    "p_tnum":           "project_turbine_count",
    "p_cap":            "project_capacity_mw",

    # Turbine‑level info
    "t_manu":           "turbine_manufacturer",
    "t_model":          "turbine_model",
    "t_cap":            "turbine_capacity_kw",
    "t_hh":             "turbine_hub_height_m",
    "t_rd":             "turbine_rotor_diameter_m",
    "t_rsa":            "turbine_rotor_swept_area_m2",
    "t_ttlh":           "turbine_total_tip_height_m",
    "t_retrofit":       "turbine_retrofit_flag",
    "t_retro_yr":       "turbine_retrofit_year",
    "t_offshore":       "offshore_flag",

    # Data quality & imagery
    "t_conf_atr":       "attribute_confidence",
    "t_conf_loc":       "location_confidence",
    "t_img_date":       "image_date",
    "t_img_src":        "image_source",
}

# apply the rename (returns a new DataFrame)
uswtdb_wind = uswtdb_wind.rename(columns=rename_map)

uswtdb_wind["zip_code"] = uswtdb_wind["zip_code"].astype(str).str.zfill(5).str.strip()

uswtdb_wind = uswtdb_wind[uswtdb_wind['state'].str.startswith('CA', na=False)].copy()
uswtdb_wind_der = uswtdb_wind[uswtdb_wind['project_capacity_mw'] <= 10].copy()
uswtdb_wind_der.to_csv('USWTDB_CA_Under_10MW.csv', index=False)

uswtdb_wind_utility = uswtdb_wind[uswtdb_wind['project_capacity_mw'] > 10].copy()

KeyError: 'zip_code'

In [6]:
# Power plant data from: https://atlas.eia.gov/datasets/eia::power-plants/explore?filters=eyJTdGF0ZSI6WyJDYWxpZm9ybmlhIl0sIkluc3RhbGxfTVciOlswLjEsNjgwOV19&location=36.025752%2C-116.778228%2C5.16
# Might have to put tags since
power_plant = pd.read_csv("Power_Plants.csv")
power_plant['County'] = power_plant['County'].str.strip().str.lower()

power_plant.loc[13284, "County"] = "sonoma"
power_plant.loc[13284, "Street_Address"] = "30 Mark West Springs Rd"
power_plant.loc[13284, "City"] = "Santa Rosa"
power_plant.loc[13284, "Zip"] = 95403

power_plant.loc[13300, "County"] = "fresno"
power_plant.loc[13300, "Street_Address"] = "S Howard Ave"
power_plant.loc[13300, "City"] = "Riverdale"
power_plant.loc[13300, "Zip"] = 93656

power_plant.loc[13305, "County"] = "fresno"
power_plant.loc[13305, "Street_Address"] = "2704 S Maple Ave"
power_plant.loc[13305, "City"] = "Fresno"
power_plant.loc[13305, "Zip"] = 93725

power_plant.loc[13307, "County"] = "tulare"
power_plant.loc[13307, "Street_Address"] = "2045 N Plaza Dr"
power_plant.loc[13307, "City"] = "Visalia"
power_plant.loc[13307, "Zip"] = 93291

power_plant.loc[13308, "County"] = "tulare"
power_plant.loc[13308, "Street_Address"] = "13213 Rd 80"
power_plant.loc[13308, "City"] = "Tipton"
power_plant.loc[13308, "Zip"] = 93272

power_plant.loc[13309, "County"] = "tulare"
power_plant.loc[13309, "Street_Address"] = "531 Poplar Ave"
power_plant.loc[13309, "City"] = "Tipton"
power_plant.loc[13309, "Zip"] = 93272

power_plant.loc[13310, "County"] = "tulare"
power_plant.loc[13310, "Street_Address"] = "531 Poplar Ave"
power_plant.loc[13310, "City"] = "Tipton"
power_plant.loc[13310, "Zip"] = 93272

power_plant.loc[13312, "County"] = "tulare"
power_plant.loc[13312, "Street_Address"] = "3240 N Plaza Dr"
power_plant.loc[13312, "City"] = "Visalia"
power_plant.loc[13312, "Zip"] = 93291

power_plant.loc[13314, "County"] = "kern"
power_plant.loc[13314, "Street_Address"] = "Zerker Rd"
power_plant.loc[13314, "City"] = "McFarland"
power_plant.loc[13314, "Zip"] = 93250

power_plant.loc[13323, "County"] = "solano"
power_plant.loc[13323, "Street_Address"] = "4451 Blum Rd"
power_plant.loc[13323, "City"] = "Martinez"
power_plant.loc[13323, "Zip"] = 94553

power_plant.loc[13324, "County"] = "kern"
power_plant.loc[13324, "Street_Address"] = "1750 E Panama Ln"
power_plant.loc[13324, "City"] = "Bakersfield"
power_plant.loc[13324, "Zip"] = 93307

power_plant.loc[13332, "County"] = "napa"
power_plant.loc[13332, "Street_Address"] = "303 Green Island Rd"
power_plant.loc[13332, "City"] = "American Canyon"
power_plant.loc[13332, "Zip"] = 94503

power_plant.loc[13338, "County"] = "san luis obispo"
power_plant.loc[13338, "Street_Address"] = "9225 N River Rd"
power_plant.loc[13338, "City"] = "San Miguel"
power_plant.loc[13338, "Zip"] = 93451

power_plant.loc[13341, "County"] = "kern"
power_plant.loc[13341, "Street_Address"] = "27125 Pond Rd"
power_plant.loc[13341, "City"] = "Wasco"
power_plant.loc[13341, "Zip"] = 93280

power_plant.loc[13427, "County"] = "fresno"
power_plant.loc[13427, "Street_Address"] = "32581 W Harlan Ave"
power_plant.loc[13427, "City"] = "Cantua Creek"
power_plant.loc[13427, "Zip"] = 93608

power_plant.loc[13445, "County"] = "merced"
power_plant.loc[13445, "Street_Address"] = "7870 Hutchins Rd"
power_plant.loc[13445, "City"] = "Dos Palos"
power_plant.loc[13445, "Zip"] = 93620

power_plant.loc[1939, "Street_Address"] = "501 Stampede Dam Road"
power_plant.loc[1939, "Zip"] = 96161


power_plant['FIPS'] = power_plant['County'].map(name_to_fips)
power_plant['FIPS'] = power_plant['FIPS'].astype(str).str.zfill(5)

rename_map = {
    "X":                "x_coord",
    "Y":                "y_coord",
    "OBJECTID":         "object_id",
    "Plant_Code":       "plant_code",
    "Plant_Name":       "plant_name",
    "Utility_ID":       "utility_id",
    "Utility_Name":     "utility_name",
    "sector_name":      "sector",
    "Street_Address":   "street_address",
    "City":             "city",
    "County":           "county",
    "State":            "state",
    "Zip":              "zip_code",
    "PrimSource":       "primary_source",
    "source_desc":      "source_description",
    "tech_desc":        "technology_description",
    "Install_MW":       "installed_capacity_mw",
    "Total_MW":         "total_capacity_mw",
    "Bat_MW":           "battery_capacity_mw",
    "Bio_MW":           "biomass_capacity_mw",
    "Coal_MW":          "coal_capacity_mw",
    "Geo_MW":           "geothermal_capacity_mw",
    "Hydro_MW":         "hydro_capacity_mw",
    "HydroPS_MW":       "hydro_pumped_storage_mw",
    "NG_MW":            "natural_gas_capacity_mw",
    "Nuclear_MW":       "nuclear_capacity_mw",
    "Crude_MW":         "crude_oil_capacity_mw",
    "Solar_MW":         "solar_capacity_mw",
    "Wind_MW":          "wind_capacity_mw",
    "Other_MW":         "other_capacity_mw",
    "Source":           "data_source",
    "Period":           "reporting_period",
    "Longitude":        "longitude",
    "Latitude":         "latitude",
    "FIPS":             "fips",
}

power_plant = power_plant.rename(columns=rename_map)
power_plant["state"] = power_plant["state"].map(state_to_abbr)
power_plant = power_plant[power_plant['state'] == "CA"].copy()
power_plant = power_plant[power_plant['total_capacity_mw'] <= 10].copy()

# plant_types = ["battery", "biomass", "hydro", "nuclear", "crude_oil", "solar", "wind", "other"] #  "natural_gas", "geothermal", "coal" removed because not DER but is plant
# total = 0
# for index, row in power_plant.iterrows():
#     current = 0
#     for type in plant_types:

#         if row[type + "_capacity_mw"] > 0:
#             current += 1

#     if current > 1:
#         print(row)
#         total += 1
# print(total)
der_sectors = [
    'IPP Non-CHP', 'Commercial Non-CHP', 'Commercial CHP',
    'Industrial Non-CHP', 'Industrial CHP', 'IPP CHP'
]

power_plant_der = power_plant[
    (power_plant['sector'].isin(der_sectors)) &
    (power_plant['total_capacity_mw'] <= 10)
]
power_plant_der["zip_code"] = power_plant_der["zip_code"].astype(str).str.zfill(5).str.strip()

power_plant_utility = power_plant[
    (power_plant['total_capacity_mw'] > 10) |
    (power_plant['sector'].isin(["Electric Utility"]))
]
print(power_plant_der.columns)

Index(['x_coord', 'y_coord', 'object_id', 'plant_code', 'plant_name',
       'utility_id', 'utility_name', 'sector', 'street_address', 'city',
       'county', 'state', 'zip_code', 'primary_source', 'source_description',
       'technology_description', 'installed_capacity_mw', 'total_capacity_mw',
       'battery_capacity_mw', 'biomass_capacity_mw', 'coal_capacity_mw',
       'geothermal_capacity_mw', 'hydro_capacity_mw',
       'hydro_pumped_storage_mw', 'natural_gas_capacity_mw',
       'nuclear_capacity_mw', 'crude_oil_capacity_mw', 'solar_capacity_mw',
       'wind_capacity_mw', 'other_capacity_mw', 'data_source',
       'reporting_period', 'longitude', 'latitude', 'fips'],
      dtype='object')


In [7]:
# Data on EV cars and chargers, https://www.energy.ca.gov/data-reports/energy-almanac/zero-emission-vehicle-and-infrastructure-statistics-collection/electric
ev_cars = pd.read_excel("Sales_MAP_County_data.xlsx")
ev_cars['County'] = ev_cars['County'].str.strip().str.lower()
ev_cars['fips'] = ev_cars['County'].map(name_to_fips)
ev_cars.rename(columns={"County":"county", "VIN":"vin"}, inplace=True)
ev_cars['fips'] = ev_cars['fips'].astype(str).str.zfill(5)
# print(ev_cars.columns)

ev_chargers = pd.read_excel('Charger_County Map_Full Data_data_zip_lat-lon.xlsx')
ev_chargers['County'] = ev_chargers['County'].str.strip().str.lower()
ev_chargers['fips'] = ev_chargers['County'].map(name_to_fips)
ev_chargers.rename(columns={"County":"county", 'Access': "access_type", 'DC Fast': "dc_fast_chargers", 'Level 1':"level1_chargers", 'Level 2':"level2_chargers",
       'Number of Chargers':"total_chargers", "ZIP":"zip_code"}, inplace=True)
ev_chargers['fips'] = ev_chargers['fips'].astype(str).str.zfill(5)
ev_chargers["zip_code"] = ev_chargers["zip_code"].astype(str).str.zfill(5).str.strip()

print(ev_chargers.head)

<bound method NDFrame.head of         county     access_type  dc_fast_chargers  level1_chargers  \
0      alameda  Shared Private               102               72   
1      alameda          Public               132                0   
2      alameda          Public                 0                0   
3      alameda          Public                 1                0   
4      alameda          Public                 1                0   
...        ...             ...               ...              ...   
18568     yuba          Public                 0                0   
18569     yuba          Public                 0                0   
18570     yuba          Public                 0                0   
18571     yuba          Public                 8                0   
18572     yuba          Public                 4                0   

       level2_chargers  total_chargers  \
0                 4040            4142   
1                  688             820   
2              

In [8]:
rename_map2 = {
    'Application Id': 'application_id',
    'Matched CSI Application Number': 'matched_california_solar_initiative_application_number',
    'Application Status': 'application_status',
    'Utility': 'utility',
    'Service City': 'service_city',
    'Service Zip': 'zip_code',
    'Service County': 'county',
    'Technology Type': 'technology_type',
    'System Size DC': 'system_size_dc',
    'System Size AC': 'system_size_ac',
    'Storage Capacity (kWh)': 'storage_capacity_kwh',
    'Storage Size (kW AC)': 'storage_size_kw_ac',
    'Inverter Size (kW AC)': 'inverter_size_kw_ac',
    'Tilt': 'tilt',
    'Azimuth': 'azimuth',
    'Mounting Method': 'mounting_method',
    'Tracking': 'tracking',
    'Customer Sector': 'sector',
    'App Received Date': 'application_received_date',
    'App Complete Date': 'application_complete_date',
    'App Approved Date': 'application_approved_date',
    'Self Installer': 'self_installer',
    'Installer Name': 'installer_name',
    'Installer Phone': 'installer_phone',
    'Installer City': 'installer_city'
}


# Project Data from: https://www.californiadgstats.ca.gov/downloads/
PGE_intercon = pd.read_csv("PGE_Interconnected_Project_Sites_2025-01-31.csv")
PGE_intercon.rename(columns=rename_map2, inplace=True)
PGE_intercon['county'] = PGE_intercon['county'].str.strip().str.lower()
PGE_intercon['fips'] = PGE_intercon['county'].map(name_to_fips)
PGE_intercon['fips'] = PGE_intercon['fips'].astype(str).str.zfill(5)
PGE_intercon["zip_code"] = PGE_intercon["zip_code"].astype(str).str.zfill(5).str.strip()

# print(PGE_intercon['FIPS'])

SCE_intercon = pd.read_csv("SCE_Interconnected_Project_Sites_2025-01-31.csv")
SCE_intercon.rename(columns=rename_map2, inplace=True)
SCE_intercon['county'] = SCE_intercon['county'].str.strip().str.lower()
SCE_intercon['fips'] = SCE_intercon['county'].map(name_to_fips)
SCE_intercon['fips'] = SCE_intercon['fips'].astype(str).str.zfill(5)
SCE_intercon["zip_code"] = SCE_intercon["zip_code"].astype(str).str.zfill(5).str.strip()


SDGE_intercon = pd.read_csv("SDGE_Interconnected_Project_Sites_2025-01-31.csv")
SDGE_intercon.rename(columns=rename_map2, inplace=True)
SDGE_intercon['county'] = SDGE_intercon['county'].str.strip().str.lower()
SDGE_intercon['fips'] = SDGE_intercon['county'].map(name_to_fips)
SDGE_intercon['fips'] = SDGE_intercon['fips'].astype(str).str.zfill(5)
SDGE_intercon["zip_code"] = SDGE_intercon["zip_code"].astype(str).str.zfill(5).str.strip()

# print(SDGE_intercon['FIPS'])

/var/folders/z3/7v87dnm11ds6n_6j078pc3qw0000gn/T/ipykernel_15869/1908426259.py:31: DtypeWarning: Columns (13,14,15,29,30,31,32,36,37,40,41,42,44,47,48,58,59,61,62,64,65,67,68,70,71,73,74,76,77,79,80,82,83,85,86,88,89,91,92,94,95,97,98,100,101,112,113,115,116,118,119,121,122,124,125,127,128,130,131,133,134,136,137,139,140,142,143,145,146,148,149,151,152,154,155,157,158,160,161,163,164,166,167,169,170,172,173) have mixed types. Specify dtype option on import or set low_memory=False.
  PGE_intercon = pd.read_csv("PGE_Interconnected_Project_Sites_2025-01-31.csv")
/var/folders/z3/7v87dnm11ds6n_6j078pc3qw0000gn/T/ipykernel_15869/1908426259.py:38: DtypeWarning: Columns (4,13,14,15,16,21,23,24,25,28,29,30,31,32,33,35,36,37,42,44,48,49,50,52,53,55,56,58,59,61,62,64,65,103,104,106,107,109,110,112,113,115,116,118,119,121,122,124,125,127,128) have mixed types. Specify dtype option on import or set low_memory=False.
  SCE_intercon = pd.read_csv("SCE_Interconnected_Project_Sites_2025-01-31.csv")
/va

In [9]:
rename_map3 = {'County':'county',
               'Latitude':"latitude",
               'Longitude':"longitude",
               'GHI (kWh/m²/day)':"ghi_kwh_m2_day",
               'FIPS':"fips",
               'Wind Speed (10m) (m/s)':"wind_speed_10m_m_s",
               'Wind Speed (50m) (m/s)':"wind_speed_50m_m_s"
}

# solar data on sunlight penetration
solar_NASA = pd.read_csv("california_solar_data.csv")
solar_NASA.rename(columns=rename_map3, inplace=True)
solar_NASA['county'] = solar_NASA['county'].str.strip().str.lower()
solar_NASA = solar_NASA.groupby(['county']).mean(numeric_only=True).reset_index()
solar_NASA['fips'] = solar_NASA['county'].map(name_to_fips)
solar_NASA['fips'] = solar_NASA['fips'].astype(str).str.zfill(5)
print(solar_NASA.columns)

# wind data giving wind speed information
wind_NASA = pd.read_csv("california_wind_data.csv")
wind_NASA.rename(columns=rename_map3, inplace=True)
wind_NASA['county'] = wind_NASA['county'].str.strip().str.lower()
wind_NASA = wind_NASA.groupby(['county']).mean(numeric_only=True).reset_index()
wind_NASA['fips'] = wind_NASA['county'].map(name_to_fips)
wind_NASA['fips'] = wind_NASA['fips'].astype(str).str.zfill(5)
print(wind_NASA.columns)

Index(['county', 'latitude', 'longitude', 'ghi_kwh_m2_day', 'fips'], dtype='object')
Index(['county', 'latitude', 'longitude', 'wind_speed_10m_m_s',
       'wind_speed_50m_m_s', 'fips'],
      dtype='object')


In [10]:
tracking_the_sun = pd.read_csv("TTS_LBNL_public_file_21-Aug-2024_all.csv")
tracking_the_sun['county'] = tracking_the_sun['zip_code'].map(zip_to_county_dict)
tracking_the_sun['fips'] = tracking_the_sun['county']
tracking_the_sun['fips'] = tracking_the_sun['fips'].astype(str).str.zfill(5)
tracking_the_sun["zip_code"] = tracking_the_sun["zip_code"].astype(str).str.zfill(5).str.strip()

print(tracking_the_sun.head)

/var/folders/z3/7v87dnm11ds6n_6j078pc3qw0000gn/T/ipykernel_15869/1168738506.py:1: DtypeWarning: Columns (1,2,3,8,11,15,16,18,20,29,32,35,38,39,40,41,42,43,44,45,46,53,54,56,57,59,60,63,64,65,66,67,68,74,75,79,80) have mixed types. Specify dtype option on import or set low_memory=False.
  tracking_the_sun = pd.read_csv("TTS_LBNL_public_file_21-Aug-2024_all.csv")


<bound method NDFrame.head of                         data_provider_1 data_provider_2  system_ID_1  \
0                               MA DOER              -1  SMAES_00001   
1                               MA DOER              -1  SMAES_00005   
2                               MA DOER              -1  SMAES_00018   
3                               MA DOER              -1  SMAES_00022   
4                               MA DOER              -1  SMAES_00190   
...                                 ...             ...          ...   
3427095  Gainesville Regional Utilities              -1           -1   
3427096  Gainesville Regional Utilities              -1           -1   
3427097  Gainesville Regional Utilities              -1           -1   
3427098  Gainesville Regional Utilities              -1           -1   
3427099  Gainesville Regional Utilities              -1           -1   

        system_ID_2 installation_date  PV_system_size_DC  \
0                -1       15-Nov-2019        

: 

In [12]:
# scaler = MinMaxScaler()

# # load US county shapefile
# us_counties = gpd.read_file('https://raw.githubusercontent.com/holtzy/The-Python-Graph-Gallery/master/static/data/US-counties.geojson')
# us_counties['id'] = us_counties['id'].astype(str).str.zfill(5)
# ca_counties = us_counties[us_counties['STATE'] == '06'].copy()
# ca_counties['FIPS'] = ca_counties['id'].copy()
# ca_project = all_intercon = pd.concat([PGE_intercon, SCE_intercon, SDGE_intercon], ignore_index=True)
# # dfs_to_merge = [ca_counties, uswtdb_wind, ca_project, solar_NASA, wind_NASA, storage, power_plant, ev_cars, ev_chargers]
# # merged_data = pd.concat(dfs_to_merge, axis=1)
# merged_data = df_ca_counties.merge(uswtdb_wind, on='FIPS', how='left', suffixes=('', '_wind'))
# merged_data = merged_data.merge(ca_counties, on='FIPS', how='left', suffixes=('', '_count'))
# merged_data = merged_data.merge(storage, left_on='id', right_on='FIPS', how='left', suffixes=('', '_storage'))
# merged_data = merged_data.merge(power_plant, left_on='id', right_on='FIPS', how='left', suffixes=('', '_powerplant'))
# merged_data = merged_data.merge(ev_cars, left_on='id', right_on='FIPS', how='left', suffixes=('', '_cars'))
# merged_data = merged_data.merge(ev_chargers, left_on='id', right_on='FIPS', how='left', suffixes=('', '_chargers'))

# # Merge remaining datasets with suffixes
# merged_data = merged_data.merge(ca_project, left_on='id', right_on='FIPS', how='left', suffixes=('', '_project'))
# merged_data = merged_data.merge(solar_NASA, left_on='id', right_on='FIPS', how='left', suffixes=('', '_solar'))
# merged_data = merged_data.merge(wind_NASA, left_on='id', right_on='FIPS', how='left', suffixes=('', '_wind_nasa'))

# # Drop redundant FIPS columns from right-side DataFrames
# merged_data = merged_data.drop(columns=[
#     'FIPS_wind', 'FIPS_storage', 'FIPS_powerplant',
#     'FIPS_cars', 'FIPS_chargers', 'FIPS_project',
#     'FIPS_solar', 'FIPS_wind_nasa', 'FIPS_count',
#     'County_wind', 'County_storage', 'County_powerplant',
#     'County_cars', 'County_chargers', 'County_project',
#     'County_solar', 'County_wind_nasa', 'County_count',
#     'Latitude_wind', 'Latitude_storage', 'Latitude_powerplant',
#     'Latitude_cars', 'Latitude_chargers', 'Latitude_project',
#     'Latitude_solar', 'Latitude_wind_nasa', 'Latitude_count',
#     'Longitude_wind', 'Longitude_storage', 'Longitude_powerplant',
#     'Longitude_cars', 'Longitude_chargers', 'Longitude_project',
#     'Longitude_solar', 'Longitude_wind_nasa',
# ], errors='ignore')

# merged_data.rename(columns={'id':'FIPS'}, inplace=True)

# # columns_to_drop = ['Area_Name', 'id', 'GEO_ID', 'NAME', 'LSAD', 'CENSUSAREA', 'FIPS_wind','State', 'Latitude_wind_NASA', 'Longitude_wind_NASA',
# #        'FIPS_solar', 'County_wind_NASA', 'Latitude_wind_NASA',
# #        'Longitude_wind_NASA']
# # merged_data = merged_data.drop(columns=columns_to_drop)

# # FOR VISUALIZING !! color columns bby cluster using this code
# for column in merged_data.columns:
#     if merged_data[column].dtype.name != "geometry":  # Skip geometry column
#         merged_data[column] = pd.to_numeric(merged_data[column], errors='coerce')
#         merged_data[column] = merged_data[column].fillna(0)
#         merged_data[column] = scaler.fit_transform(merged_data[[column]])
#         fig, ax = plt.subplots(1, 1, figsize=(12, 8))
#         merged_data.plot(column=column, cmap='viridis', linewidth=0.5, edgecolor='white', ax=ax, legend=True,
#                     legend_kwds={'shrink': 0.75})
#         ax.axis('off')
#         ax.set_title(f'California Counties - {column}', fontsize=16)
#         plt.tight_layout()
#         plt.show()

# merged_data = merged_data.loc[:, ~merged_data.columns.duplicated()].copy()
# merged_data['FIPS'] = pd.to_numeric(merged_data['FIPS'], errors='coerce')
# df_sorted = merged_data.sort_values(by='FIPS', ascending=True).reset_index(drop=True)
# merged_data['FIPS'] = merged_data['FIPS'].astype(str).str.zfill(5)
# print(merged_data.head)
# merged_data.to_csv('california_energy_data.csv', index=False)

# print()

: 

In [5]:
# List of dataframes and their names for prefixing
datasets = {
    "tracking_the_sun": tracking_the_sun,
    "SDGE_intercon": SDGE_intercon,
    "SCE_intercon": SCE_intercon,
    "PGE_intercon": PGE_intercon,
    "ev_chargers": ev_chargers,
    "power_plant_der": power_plant_der,
    "uswtdb_wind": uswtdb_wind,
    "storage_der": storage_der,
}

# Standardize and prefix columns (except for 'zip_code')
for name, df in datasets.items():
    df.columns = [
        f"{name}__{col}" if col != "zip_code" else col
        for col in df.columns
    ]
    datasets[name] = df

# Merge all dataframes on 'zip_code' using outer join
merged_df = None
for df in datasets.values():
    if merged_df is None:
        merged_df = df
    else:
        merged_df = pd.merge(merged_df, df, on="zip_code", how="outer")

# Now `merged_df` contains the combined result


NameError: name 'tracking_the_sun' is not defined

In [38]:
"""
import requests
import pandas as pd
import geopandas as gpd

# Define the date range
start_date = "2024-01-01"
end_date = "2024-12-31"

# Load ZIP Code Tabulation Areas (ZCTAs)
zip_data = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2021/ZCTA520/tl_2021_us_zcta520.zip")

# Optionally filter for only California ZIPs using an external mapping of ZIPs to state (not in shapefile)
# You could instead merge with a ZIP-to-state crosswalk
# For example: https://www.huduser.gov/portal/datasets/usps_crosswalk.html

# Filter for ZIPs that start with California prefixes (90–96)
zip_data["ZCTA5CE20"] = zip_data["ZCTA5CE20"].astype(str)
california_zips = zip_data[zip_data["ZCTA5CE20"].str.startswith(tuple(str(i) for i in range(900, 967)))].copy()

# Get centroid for each ZIP code
california_zips["Latitude"] = california_zips["geometry"].centroid.y
california_zips["Longitude"] = california_zips["geometry"].centroid.x

# Store wind data
wind_data_list = []

# Define function to fetch wind data from NASA POWER
def get_wind_data_zip(lat, lon, zip_code):
    url = f"https://power.larc.nasa.gov/api/temporal/daily/point?parameters=WS10M,WS50M&community=RE&longitude={lon}&latitude={lat}&start={start_date.replace('-', '')}&end={end_date.replace('-', '')}&format=JSON"

    response = requests.get(url)
    data = response.json()

    if "properties" in data and "parameter" in data["properties"]:
        wind_10m = data["properties"]["parameter"]["WS10M"]
        wind_50m = data["properties"]["parameter"]["WS50M"]

        for date in wind_10m.keys():
            wind_data_list.append({
                "ZIP Code": zip_code,
                "Date": date,
                "Latitude": lat,
                "Longitude": lon,
                "Wind Speed (10m) (m/s)": wind_10m[date],
                "Wind Speed (50m) (m/s)": wind_50m[date]
            })
    else:
        print(f"❌ Error fetching data for ZIP {zip_code}")

# Loop through ZIPs (you can limit for testing)
for _, row in california_zips.iterrows():
    get_wind_data_zip(row["Latitude"], row["Longitude"], row["ZCTA5CE20"])

# Convert to DataFrame and save
df = pd.DataFrame(wind_data_list)
df["Date"] = pd.to_datetime(df["Date"])
df.to_csv("california_zip_wind_data.csv", index=False)
print("✅ Wind speed data by ZIP code saved!")
"""

/var/folders/z3/7v87dnm11ds6n_6j078pc3qw0000gn/T/ipykernel_1216/2791011850.py:21: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  california_zips["Latitude"] = california_zips["geometry"].centroid.y
/var/folders/z3/7v87dnm11ds6n_6j078pc3qw0000gn/T/ipykernel_1216/2791011850.py:22: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  california_zips["Longitude"] = california_zips["geometry"].centroid.x


❌ Error fetching data for ZIP 95123
✅ Wind speed data by ZIP code saved!


In [ ]:
"""
import requests
import pandas as pd
import geopandas as gpd

# Define date range
start_date = "2024-01-01"
end_date = "2024-12-31"

# Load ZCTA (ZIP Code Tabulation Area) boundaries
zcta_data = gpd.read_file("https://www2.census.gov/geo/tiger/TIGER2021/ZCTA520/tl_2021_us_zcta520.zip")

# Filter for California ZIP codes (starting with 90–96)
zcta_data["ZCTA5CE20"] = zcta_data["ZCTA5CE20"].astype(str)
zcta_ca = zcta_data[zcta_data["ZCTA5CE20"].str.startswith(tuple(str(i) for i in range(900, 967)))]

# Compute centroid for each ZIP area
zcta_ca["Latitude"] = zcta_ca["geometry"].centroid.y
zcta_ca["Longitude"] = zcta_ca["geometry"].centroid.x

# Initialize list for solar data
solar_data_list = []

# Function to fetch GHI from NASA POWER API
def get_solar_data(lat, lon, zip_code):
    url = (
        f"https://power.larc.nasa.gov/api/temporal/daily/point?"
        f"parameters=ALLSKY_SFC_SW_DWN&community=RE&longitude={lon}"
        f"&latitude={lat}&start={start_date.replace('-', '')}"
        f"&end={end_date.replace('-', '')}&format=JSON"
    )

    response = requests.get(url)
    data = response.json()

    if "properties" in data and "parameter" in data["properties"]:
        solar_data = data["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"]

        for date, ghi in solar_data.items():
            solar_data_list.append({
                "ZIP Code": zip_code,
                "Date": date,
                "Latitude": lat,
                "Longitude": lon,
                "GHI (kWh/m²/day)": ghi
            })
    else:
        print(f"❌ Error fetching data for ZIP {zip_code}")

# Loop through ZCTA centroids
for _, row in zcta_ca.iterrows():
    get_solar_data(row["Latitude"], row["Longitude"], row["ZCTA5CE20"])

# Save as DataFrame
df = pd.DataFrame(solar_data_list)
df["Date"] = pd.to_datetime(df["Date"])
df.to_csv("california_zip_solar_data.csv", index=False)

print("✅ Solar data for California ZIP codes saved to 'california_zip_solar_data.csv'!")
"""

In [8]:
# import requests
# import pandas as pd
# import time
# import io

# # NSRDB API Key (Get yours from https://developer.nrel.gov/signup/)
# API_KEY = "bHxtfGBdKfYGebiXeoSUVWeemSSLz8A2JkeD1VGC"

# # List of all 58 California counties with representative latitude & longitude
# california_counties = [
#     {"County": "Alameda", "Lat": 37.6500, "Lon": -121.9000},
#     {"County": "Alpine", "Lat": 38.5968, "Lon": -119.8223},
#     {"County": "Amador", "Lat": 38.3482, "Lon": -120.7742},
#     {"County": "Butte", "Lat": 39.6253, "Lon": -121.5370},
#     {"County": "Calaveras", "Lat": 38.1960, "Lon": -120.6800},
#     {"County": "Colusa", "Lat": 39.1780, "Lon": -122.2370},
#     {"County": "Contra Costa", "Lat": 37.8534, "Lon": -121.9018},
#     {"County": "Del Norte", "Lat": 41.7423, "Lon": -124.1830},
#     {"County": "El Dorado", "Lat": 38.7426, "Lon": -120.4358},
#     {"County": "Fresno", "Lat": 36.7378, "Lon": -119.7871},
#     {"County": "Glenn", "Lat": 39.5994, "Lon": -122.3933},
#     {"County": "Humboldt", "Lat": 40.7450, "Lon": -123.8695},
#     {"County": "Imperial", "Lat": 32.8490, "Lon": -115.5690},
#     {"County": "Inyo", "Lat": 36.5112, "Lon": -117.4032},
#     {"County": "Kern", "Lat": 35.4937, "Lon": -118.8591},
#     {"County": "Kings", "Lat": 36.0758, "Lon": -119.8173},
#     {"County": "Lake", "Lat": 39.0836, "Lon": -122.7633},
#     {"County": "Lassen", "Lat": 40.6228, "Lon": -120.5770},
#     {"County": "Los Angeles", "Lat": 34.0522, "Lon": -118.2437},
#     {"County": "Madera", "Lat": 37.2519, "Lon": -119.6963},
#     {"County": "Marin", "Lat": 38.0834, "Lon": -122.7633},
#     {"County": "Mariposa", "Lat": 37.5004, "Lon": -119.9657},
#     {"County": "Mendocino", "Lat": 39.5500, "Lon": -123.4384},
#     {"County": "Merced", "Lat": 37.3022, "Lon": -120.4820},
#     {"County": "Modoc", "Lat": 41.5700, "Lon": -120.7000},
#     {"County": "Mono", "Lat": 38.0000, "Lon": -119.0000},
#     {"County": "Monterey", "Lat": 36.6002, "Lon": -121.8947},
#     {"County": "Napa", "Lat": 38.5025, "Lon": -122.2654},
#     {"County": "Nevada", "Lat": 39.3030, "Lon": -120.7459},
#     {"County": "Orange", "Lat": 33.7175, "Lon": -117.8311},
#     {"County": "Placer", "Lat": 39.0916, "Lon": -120.8039},
#     {"County": "Plumas", "Lat": 39.9275, "Lon": -120.7381},
#     {"County": "Riverside", "Lat": 33.9533, "Lon": -117.3961},
#     {"County": "Sacramento", "Lat": 38.5816, "Lon": -121.4944},
#     {"County": "San Benito", "Lat": 36.5761, "Lon": -121.0024},
#     {"County": "San Bernardino", "Lat": 34.1083, "Lon": -117.2898},
#     {"County": "San Diego", "Lat": 32.7157, "Lon": -117.1611},
#     {"County": "San Francisco", "Lat": 37.7749, "Lon": -122.4194},
#     {"County": "San Joaquin", "Lat": 37.9577, "Lon": -121.2908},
#     {"County": "San Luis Obispo", "Lat": 35.2828, "Lon": -120.6596},
#     {"County": "San Mateo", "Lat": 37.5629, "Lon": -122.3255},
#     {"County": "Santa Barbara", "Lat": 34.4208, "Lon": -119.6982},
#     {"County": "Santa Clara", "Lat": 37.3541, "Lon": -121.9552},
#     {"County": "Santa Cruz", "Lat": 36.9741, "Lon": -122.0308},
#     {"County": "Shasta", "Lat": 40.5870, "Lon": -122.3917},
#     {"County": "Sierra", "Lat": 39.5800, "Lon": -120.5200},
#     {"County": "Siskiyou", "Lat": 41.5905, "Lon": -122.5403},
#     {"County": "Solano", "Lat": 38.3105, "Lon": -121.9018},
#     {"County": "Sonoma", "Lat": 38.5783, "Lon": -122.5797},
#     {"County": "Stanislaus", "Lat": 37.5091, "Lon": -120.9993},
#     {"County": "Sutter", "Lat": 39.0342, "Lon": -121.6739},
#     {"County": "Tehama", "Lat": 40.0982, "Lon": -122.1746}]

# year = 2020  # change as needed

# BASE_URL = "https://developer.nrel.gov/api/nsrdb/v2/solar/psm3-download.csv"

# all_data = []

# # Loop through each county and fetch data
# for county in california_counties:
#     lat, lon = county["Lat"], county["Lon"]
#     params = {
#         "api_key": API_KEY,
#         "email": "daniknut@mit.edu",  # REQUIRED: Use your registered email
#         "wkt": f"POINT({lon} {lat})",
#         "names": 2020,  # Change to a valid year (1998-2020)
#         "leap_day": "false",
#         "interval": 60,
#         "utc": "false",
#         "attributes": "wind_speed,dhi,dni,ghi"
#     }

#     # print(f"Fetching data for {county['County']} County...")

#     response = requests.get(BASE_URL, params=params)
#     if response.status_code == 200:
#         df = pd.read_csv(io.StringIO(response.text), skiprows=2)  # Corrected
#         df["County"] = county["County"]
#         df["Latitude"] = lat
#         df["Longitude"] = lon

#         all_data.append(df)
#     else:
#         print(f"Error fetching data for {county['County']}: {response.status_code}")
#         break

#     # Avoid hitting API limits
#     time.sleep(1)

# final_df = pd.concat(all_data, ignore_index=True)

# # Save to CSV
# final_df.to_pickle("california_wind_solar_data.pkl")
# print("Data saved to 'california_wind_solar_data.csv'")
# print(final_df.head)

In [9]:
# import pandas as pd
# import reverse_geocode

# # Define metadata from NSRDB dataset
# metadata = {
#     "Source": "NSRDB",
#     "Location ID": 137337,
#     "Latitude": 37.77,
#     "Longitude": -122.42,
#     "Time Zone": -8,
#     "Elevation": 32,
#     "Version": "4.0.1"
# }

# # Function to get county from latitude and longitude
# def get_county(lat, lon):
#     location = reverse_geocode.search([(lat, lon)])[0]  # Returns a dictionary
#     county = location.get("county", "Unknown")
#     state = location.get("state", "Unknown")
#     city = location.get("city", "Unknown")
#     country = location.get("country", "Unknown")
#     return county, city, state, country

# county, city, state, country = get_county(metadata["Latitude"], metadata["Longitude"])
# df = pd.read_csv("wind_speed_and_direction.csv")  # Replace with the actual file
# print(df.columns)
# df.columns = ["Year", "Month", "Day", "Hour", "Minute", "Temperature", "Wind Speed", "Wind Direction"]
# df["Datetime"] = pd.to_datetime(df[["Year", "Month", "Day", "Hour", "Minute"]])
# df["County"] = county
# df["City"] = city
# df["State"] = state
# df["Country"] = country
# df["Source"] = metadata["Source"]
# df["Location ID"] = metadata["Location ID"]
# df["Latitude"] = metadata["Latitude"]
# df["Longitude"] = metadata["Longitude"]
# df["Time Zone"] = metadata["Time Zone"]
# df["Elevation"] = metadata["Elevation"]
# df["Version"] = metadata["Version"]
# df = df[
#     ["Datetime", "Year", "Month", "Day", "Hour", "Minute", "County", "City", "State", "Country",
#      "Source", "Location ID", "Latitude", "Longitude", "Time Zone", "Elevation", "Temperature",
#      "Wind Speed", "Wind Direction"]
# ]

# df.to_csv("wind_data.csv", index=False)
# print("Processing complete. Data saved to 'processed_nsrdb_data.csv'.")